# 📊 Exploratory Data Analysis (EDA) Notebook
**Semi-Automated Classification Workflow**

Supported use cases:
- Customer Churn Prediction
- Credit Risk Prediction
- Loan Default Prediction

> **Goal:** Understand the dataset and identify issues before cleaning and modeling.

---

## Section 1 — Load Libraries

Load all required libraries for EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully.')

---
## Section 2 — Load Dataset

Upload your CSV file using the file picker below, then load it into a DataFrame.

In [ ]:
from google.colab import files

# Upload CSV file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print(f'File loaded: {filename}')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Section 3 — Dataset Preview

Quickly inspect the content of the dataset.

In [ ]:
print('--- First 5 Rows ---')
display(df.head())

print('\n--- Last 5 Rows ---')
display(df.tail())

print('\n--- Random Sample (5 rows) ---')
display(df.sample(5, random_state=42))

---
## Section 4 — Dataset Information

Understand the structure of the dataset: data types, columns, and row count.

In [ ]:
print(f'Number of Rows    : {df.shape[0]}')
print(f'Number of Columns : {df.shape[1]}')

print('\n--- Data Types ---')
print(df.dtypes.to_string())

print('\n--- Dataset Info ---')
df.info()

---
## Section 5 — Missing Value Analysis

Identify columns with missing values. This helps prioritize what to fix during cleaning.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing (%)': missing_pct
})

# Keep only columns that have at least 1 missing value
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing (%)', ascending=False)

if missing_df.empty:
    print('No missing values found.')
else:
    print(f'Columns with missing values: {len(missing_df)}')
    display(missing_df.style.format({'Missing (%)': '{:.2f}%'}))

---
## Section 6 — Duplicate Analysis

Find rows that are exact duplicates. Duplicates can distort model training.

In [ ]:
total_duplicates = df.duplicated().sum()
duplicate_pct = (total_duplicates / len(df)) * 100

print(f'Total Duplicate Rows : {total_duplicates}')
print(f'Duplicate Percentage : {duplicate_pct:.2f}%')

if total_duplicates > 0:
    print('\n⚠️  Duplicates detected. Consider removing them in the Cleaning step.')
else:
    print('\n✅ No duplicates found.')

---
## Section 7 — Target Distribution Analysis

Check how balanced the target classes are. Imbalanced targets can affect model performance.

> **Action required:** Set your target column name in the cell below.

In [ ]:
# ✏️ Set your target column name here
TARGET_COLUMN = 'Churn'  # <-- Change this to your actual target column

# -----------------------------------------------
if TARGET_COLUMN not in df.columns:
    print(f'ERROR: Column "{TARGET_COLUMN}" not found in dataset.')
    print(f'Available columns: {list(df.columns)}')
else:
    counts = df[TARGET_COLUMN].value_counts()
    percentages = df[TARGET_COLUMN].value_counts(normalize=True) * 100

    target_df = pd.DataFrame({
        'Class': counts.index,
        'Count': counts.values,
        'Percentage (%)': percentages.values
    })

    print(f'Target Column: {TARGET_COLUMN}')
    display(target_df.style.format({'Percentage (%)': '{:.2f}%'}))

    # Plot
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.countplot(x=TARGET_COLUMN, data=df, palette='Set2', ax=ax)
    ax.set_title(f'Target Distribution: {TARGET_COLUMN}')
    ax.set_xlabel(TARGET_COLUMN)
    ax.set_ylabel('Count')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=11)
    plt.tight_layout()
    plt.show()

    # Imbalance check: minority class < 30% is considered imbalanced
    min_pct = percentages.min()
    if min_pct < 30:
        print(f'\n⚠️  Status: IMBALANCED  (minority class = {min_pct:.1f}%)')
    else:
        print(f'\n✅ Status: BALANCED  (minority class = {min_pct:.1f}%)')

---
## Section 8 — Feature Type Analysis

Separate features into numerical, categorical, and datetime types. Each type requires different handling during cleaning and modeling.

In [ ]:
numerical_cols  = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
datetime_cols   = df.select_dtypes(include=['datetime64']).columns.tolist()

print(f'Numerical Features  ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical Features ({len(categorical_cols)}): {categorical_cols}')
print(f'Datetime Features   ({len(datetime_cols)}): {datetime_cols}')

---
## Section 9 — High Cardinality Check

Find columns with many unique values (e.g., CustomerID, Email, Phone). These columns usually need to be dropped or encoded specially.

In [ ]:
# High cardinality threshold: more than 50 unique values
CARDINALITY_THRESHOLD = 50

unique_counts = df.nunique().sort_values(ascending=False)

high_card_df = pd.DataFrame({
    'Feature': unique_counts.index,
    'Unique Count': unique_counts.values
})

high_card_df = high_card_df[high_card_df['Unique Count'] > CARDINALITY_THRESHOLD]

if high_card_df.empty:
    print(f'No columns with more than {CARDINALITY_THRESHOLD} unique values found.')
else:
    print(f'High cardinality columns (> {CARDINALITY_THRESHOLD} unique values):')
    display(high_card_df.reset_index(drop=True))

---
## Section 10 — Outlier Detection

Detect potential outliers in numerical columns using the **IQR method**.

A value is an outlier if it falls below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR`.

In [ ]:
outlier_results = []

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_pct = (outlier_count / len(df)) * 100

    outlier_results.append({
        'Feature': col,
        'Outlier Count': outlier_count,
        'Outlier (%)': round(outlier_pct, 2)
    })

outlier_df = pd.DataFrame(outlier_results)
outlier_df = outlier_df[outlier_df['Outlier Count'] > 0].sort_values('Outlier Count', ascending=False)

if outlier_df.empty:
    print('No outliers detected in any numerical column.')
else:
    print(f'Columns with detected outliers: {len(outlier_df)}')
    display(outlier_df.reset_index(drop=True).style.format({'Outlier (%)': '{:.2f}%'}))

---
## Section 11 — EDA Summary

A quick summary of all findings from this EDA notebook.

In [ ]:
# ---- Gather summary values ----

# Missing values
missing_features = df.isnull().sum()
missing_features = missing_features[missing_features > 0]
n_missing = len(missing_features)

# Duplicates
n_duplicates = df.duplicated().sum()

# Target status
try:
    min_pct = df[TARGET_COLUMN].value_counts(normalize=True).min() * 100
    target_status = 'BALANCED' if min_pct >= 30 else f'IMBALANCED (minority = {min_pct:.1f}%)'
except:
    target_status = 'Not checked (TARGET_COLUMN not set)'

# High cardinality
high_card_features = df.nunique()
high_card_features = high_card_features[high_card_features > CARDINALITY_THRESHOLD]
n_high_card = len(high_card_features)

# Outliers
try:
    n_outlier_features = len(outlier_df)
except:
    n_outlier_features = 'Not computed'

# ---- Print Summary ----
print('=' * 50)
print('              EDA SUMMARY')
print('=' * 50)
print(f'  Dataset Shape              : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'  Missing Value Features     : {n_missing}')
print(f'  Duplicate Rows             : {n_duplicates}')
print(f'  Target Status              : {target_status}')
print(f'  Numerical Features         : {len(numerical_cols)}')
print(f'  Categorical Features       : {len(categorical_cols)}')
print(f'  Datetime Features          : {len(datetime_cols)}')
print(f'  High Cardinality Features  : {n_high_card}')
print(f'  Features With Outliers     : {n_outlier_features}')
print('=' * 50)
print()
print('  ✅ Next Step: Run Cleaning Notebook')
print('=' * 50)